# Stereotype Classification on LLM-Generated Dutch Sentences

Part of a group project that reproduced a 2025 master's thesis on gender bias in Dutch text
generation by LLaMA models. **This notebook contains my part only.**

**Classification:** sentences generated by LLaMA 3 pair a gendered noun with an adjective. A sentence is
*stereotypical* when the noun and the adjective have the same gender, otherwise *anti-stereotypical*.
Multilingual DistilBERT is fine-tuned to tell the two apart, with two hyperparameter sets from the
thesis' search ranges.

**Extra experiment:** continue the masked-language-model pre-training of DistilBERT on a corpus in
which gendered words are replaced by a neutral `[PERSON]` token, then fine-tune again in the same way.

Outputs are from the original run on Google Colab.

## 1. Data

In [ ]:
import pandas as pd

# Cleaned sentences generated by several LLMs (course data, not included in this repository)
df = pd.read_csv("sentences_cleaned.csv")
df.head(3)

,word,sentence,model,noun_gender,adjective_gender,temperature,total_runs,male_count,female_count,male_matches,female_matches,total_in_lists
0,lamme,De lamme man was een vrolijke kerel.,llama3:text,male,male,0.5,75,1,0,['lamme'],[],1
1,impopulair,De impopulaire jongen heeft het niet gemakkelijk.,llama3:text,male,male,0.5,75,1,0,['impopulaire'],[],1
2,overmoedig,Overmoedig is de man die je bent.,llama3:text,male,male,0.5,75,1,0,['overmoedig'],[],1


In [ ]:
# Keep only the sentences generated by LLaMA 3
df = df[df["model"] == "llama3:text"].copy()

# Label: 1 = stereotypical (noun and adjective have the same gender), 0 = anti-stereotypical
df["label"] = (df["noun_gender"] == df["adjective_gender"]).astype(int)

# Keep the required columns, drop missing values and duplicate sentences
df = df[["sentence", "label"]].dropna().drop_duplicates(subset=["sentence"])
df.head(10)

,sentence,label
0,De lamme man was een vrolijke kerel.,1
1,De impopulaire jongen heeft het niet gemakkelijk.,1
2,Overmoedig is de man die je bent.,1
3,Hij was vooraanstaand in het leven van een man...,1
4,"Deze gewetenloze man, mannen, jongen, jongens,...",1
5,"De man, mannen, jongen, jongens, heer, heren, ...",1
6,"Dit is een schatrijk man, mannen, jongen, jong...",1
7,"Deze lamme man, mannen, jongen, jongens, heer,...",1
8,"Dit is een lucratief man, mannen, jongen, jong...",1
9,"Deze planmatige man, mannen, jongen, jongens, ...",1


## 2. Train / validation / test split (70 / 15 / 15)

In [ ]:
# 3.2. partition the sentences into 70% training, 15% validation, and 15% test (random state 111, stratified by label)
from sklearn.model_selection import train_test_split

train_idx, valtest_idx = train_test_split(df.index, test_size=0.30, random_state=111, stratify=df["label"])
val_idx, test_idx = train_test_split(valtest_idx, test_size=0.50, random_state=111, stratify=df.loc[valtest_idx, "label"])

train_df = df.loc[train_idx].reset_index(drop=True)
val_df   = df.loc[val_idx].reset_index(drop=True)
test_df  = df.loc[test_idx].reset_index(drop=True)

n = len(df)
print(f"Train: {len(train_df)} ({len(train_df)/n:.2f})")
print(f"Val  : {len(val_df)} ({len(val_df)/n:.2f})")
print(f"Test : {len(test_df)} ({len(test_df)/n:.2f})")

Train: 2775 (0.70)
Val  : 595 (0.15)
Test : 595 (0.15)

## 3. Fine-tuning DistilBERT

In [ ]:
# Fine-tune multilingual DistilBERT to tell stereotypical from anti-stereotypical sentences

import os
# Disable external logging
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Model configuration
model_name = "distilbert-base-multilingual-cased"
num_labels = 2
tok = AutoTokenizer.from_pretrained(model_name)

# Convert DataFrames to tokenized Datasets
train_ds = Dataset.from_pandas(train_df[["sentence","label"]]).map(
    lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=128), batched=True)
val_ds = Dataset.from_pandas(val_df[["sentence","label"]]).map(
    lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=128), batched=True)
test_ds = Dataset.from_pandas(test_df[["sentence","label"]]).map(
    lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=128), batched=True)

# Two hyperparameter sets inside the search ranges reported in the reproduced thesis
hp_sets = {
    "SET_A": dict(learning_rate=2e-5, per_device_train_batch_size=16, weight_decay=0.01, num_train_epochs=3),
    "SET_B": dict(learning_rate=4e-5, per_device_train_batch_size=32, weight_decay=0.0001, num_train_epochs=3),
}

# Evaluate and store results
results = {}
for name, hp in hp_sets.items():
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    args = TrainingArguments(output_dir=f"out_{name}", seed=42, eval_strategy="epoch", disable_tqdm=False,
                  logging_strategy="epoch", save_strategy="no", report_to="none", **hp)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
               compute_metrics=lambda p: {"f1": f1_score(p.label_ids, p.predictions.argmax(-1), pos_label=1)})
    trainer.train()

    results[name] = {"val_f1": float(trainer.evaluate(val_ds)["eval_f1"]),
                     "test_f1": float(trainer.evaluate(test_ds)["eval_f1"])}
    print(f"{name}: val_f1={results[name]['val_f1']:.4f}, test_f1={results[name]['test_f1']:.4f}\n")

# The set with the higher validation F1 is the final model
best_set_name = max(results, key=lambda k: results[k]["val_f1"])
final_test_f1 = results[best_set_name]["test_f1"]
print(f"Best Model is: {best_set_name} with test F1 = {results[best_set_name]['test_f1']:.4f}")

{'loss': 0.696, 'grad_norm': 2.976231098175049, 'learning_rate': 1.3371647509578544e-05, 'epoch': 1.0}
{'eval_loss': 0.6893570423126221, 'eval_f1': 0.6454081632653061, 'eval_runtime': 2.1861, 'eval_samples_per_second': 272.179, 'eval_steps_per_second': 34.308, 'epoch': 1.0}
{'loss': 0.6027, 'grad_norm': 13.304296493530273, 'learning_rate': 6.7049808429118775e-06, 'epoch': 2.0}
{'eval_loss': 0.5361136198043823, 'eval_f1': 0.7526555386949925, 'eval_runtime': 2.025, 'eval_samples_per_second': 293.831, 'eval_steps_per_second': 37.037, 'epoch': 2.0}
{'loss': 0.4074, 'grad_norm': 39.480735778808594, 'learning_rate': 3.831417624521073e-08, 'epoch': 3.0}
{'eval_loss': 0.464193195104599, 'eval_f1': 0.7792642140468228, 'eval_runtime': 2.1113, 'eval_samples_per_second': 281.815, 'eval_steps_per_second': 35.523, 'epoch': 3.0}
{'train_runtime': 103.5154, 'train_samples_per_second': 80.423, 'train_steps_per_second': 5.043, 'train_loss': 0.568706950922122, 'epoch': 3.0}
{'eval_loss': 0.46419319510459

Hyperparameter search ranges reported in the thesis:

- Learning rate: [1e-5, 5e-5]
- Batch size: {8, 16, 32}
- Weight decay: [1e-6, 1e-2]

Inputs are padded to 128 tokens, the seed is 42 and every run trains for 3 epochs.

| Set | Learning rate | Batch size | Weight decay | Test F1 |
|---|---|---|---|---|
| A | 2e-5 | 16 | 0.01 | **0.778** |
| B | 4e-5 | 32 | 0.0001 | 0.762 |

Set A has the higher validation F1 and is used below.

## 4. Extra experiment: pre-training on a gender-neutral corpus

In [ ]:
# Build a gender-neutral corpus from the training sentences by replacing gendered words with [PERSON].
# Notes: the text is lower-cased although the model is cased, and "[PERSON]" is not added to the
# vocabulary, so the tokenizer splits it into word pieces. Kept as in the original run.
male_words   = ["man","mannen","jongen","jongens","heer","heren","vader","vaders","hij","hem","zijn","z'n"]
female_words = ["vrouw","vrouwen","meisje","meisjes","dame","dames","moeder","moeders","zij","haar","ze"]

neutral = train_df["sentence"].astype(str).str.lower()
for w in male_words:   neutral = neutral.str.replace(rf"\b{w}\b", "[PERSON]", regex=True)
for w in female_words: neutral = neutral.str.replace(rf"\b{w}\b", "[PERSON]", regex=True)

mlm_texts = neutral.dropna().drop_duplicates().tolist()[:10000]
print("[INFO] Neutral corpus size:", len(mlm_texts))

# Continue masked-language-model pre-training on the neutral corpus for 3 epochs
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments

base_model = "distilbert-base-multilingual-cased"
tok = AutoTokenizer.from_pretrained(base_model)

mlm_ds = Dataset.from_dict({"text": mlm_texts}).map(
    lambda b: tok(b["text"], truncation=True, padding="max_length", max_length=128), batched=True)

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=0.15)
mlm_model = AutoModelForMaskedLM.from_pretrained(base_model)

mlm_args = TrainingArguments(
    output_dir="dapt_mlm",
    per_device_train_batch_size=32,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    seed=42,
    disable_tqdm=False,
    report_to="none"
)
Trainer(model=mlm_model, args=mlm_args, train_dataset=mlm_ds, data_collator=collator).train()
mlm_model.save_pretrained("dapt_mlm"); tok.save_pretrained("dapt_mlm")

# Prepare datasets (same split as Task 3)
train_ds = Dataset.from_pandas(train_df[["sentence","label"]]).map(
    lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=128), batched=True)
val_ds   = Dataset.from_pandas(val_df[["sentence","label"]]).map(
    lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=128), batched=True)
test_ds  = Dataset.from_pandas(test_df[["sentence","label"]]).map(
    lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=128), batched=True)

# Fine-tune exactly as in Task 3 with SET_A and report the test F1
lr, train_bs, wd, epochs = 2e-5, 16, 0.01, 3

model = AutoModelForSequenceClassification.from_pretrained("dapt_mlm", num_labels=2)
clf_args = TrainingArguments(
    output_dir="out_DAPT_SET_A",
    learning_rate=lr,
    per_device_train_batch_size=train_bs,
    weight_decay=wd,
    num_train_epochs=epochs,
    seed=42,
    report_to="none"
)
trainer = Trainer(
    model=model, args=clf_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=lambda p: {"f1": f1_score(p[1], p[0].argmax(-1), pos_label=1)}
)
trainer.train()

dapt_test_f1 = float(trainer.evaluate(test_ds)["eval_f1"])
print("DAPT Test F1 =", f"{dapt_test_f1:.4f}")

[INFO] Neutral corpus size: 2751

Step,Training Loss


{'loss': 0.5311, 'grad_norm': 12.832633972167969, 'learning_rate': 8.812260536398468e-07, 'epoch': 2.873563218390805}
{'train_runtime': 188.4212, 'train_samples_per_second': 44.183, 'train_steps_per_second': 2.77, 'train_loss': 0.523002951537969, 'epoch': 3.0}
{'eval_loss': 0.4384966194629669, 'eval_f1': 0.8060200668896321, 'eval_runtime': 2.1545, 'eval_samples_per_second': 276.165, 'eval_steps_per_second': 34.811, 'epoch': 3.0}
DAPT Test F1 = 0.8060

## Result

| Model | Test F1 |
|---|---|
| DistilBERT, fine-tuned (Set A) | 0.778 |
| DistilBERT, extra pre-training on the neutral corpus, then fine-tuned (Set A) | **0.806** |

Test F1 rose from 0.778 to 0.806. Two caveats: the neutral corpus was built from the training
sentences themselves rather than from an independent corpus, and each setting was run once, so
the gain may partly be run-to-run variation.